# 10 - Dashboard export (WA-TGCN v2)

Regenerates `dashboard_export/{stations,playback,summary}.json` from the **v2**
model (notebook 09) so `app.py` shows v2 with no code change.

- Reuses 09's exact 26-feature pipeline and the forecast-climatology graph.
- Loads the **saved** v2 scalers / checkpoints / `v2_clim_grid.npy` - nothing is refit.
- Backs up the old v1 export to `dashboard_export_v1/` on first run.

Run top to bottom (~3 min, inference only), then `streamlit run app.py`.

In [17]:
import os, json, pickle, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR  = os.path.join(OUT, 'scalers')
MODEL_DIR   = os.path.join(OUT, 'models')
RESULTS_DIR = os.path.join(OUT, 'results')
DASH        = os.path.join(V2_DIR, 'dashboard_export')

DATASET_CSV      = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
DYNAMIC_ADJ_PATH = os.path.join(V2_DIR, 'dynamic_adj_matrix.npy')
STATIONS_EXCEL   = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
CLIM_GRID_PATH   = os.path.join(V2_DIR, 'v2_clim_grid.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CPCB_THRESHOLD, WHO_THRESHOLD = 60.0, 15.0
N_PLAYBACK = 970
INPUT_WINDOW, HORIZON = 48, 24
THRESHOLD_KM = 12.0

cfg = json.load(open(os.path.join(MODEL_DIR, 'v2_config.json')))
FEATURE_COLS = cfg['feature_cols']; TIME_COLS = cfg['time_cols']
F = cfg['in_features']; N_CFG = cfg['n_stations']; SEEDS = cfg['seeds']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']

bk = os.path.join(V2_DIR, 'dashboard_export_v1')
if not os.path.isdir(bk):
    shutil.copytree(DASH, bk); print(f'backed up v1 export -> {bk}')
else:
    print(f'v1 backup already present -> {bk}')
print(f'Device {DEVICE} | F={F} | {len(SEEDS)} seeds {SEEDS}')

# HOW TO CHECK: prints the backup path and 'cuda'. F should be 26.

v1 backup already present -> D:\FYP\Eco-Twin\v2_clean\dashboard_export_v1
Device cuda | F=26 | 7 seeds [42, 7, 123, 2024, 999, 2025, 77]


In [18]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)

df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(station_names)
assert N == N_CFG, (N, N_CFG)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
dyn_adj = np.load(DYNAMIC_ADJ_PATH).astype(np.float32)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arctan2(np.sqrt(a), np.sqrt(1-a))

dist_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            dist_matrix[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])
within_threshold = (dist_matrix > 0) & (dist_matrix <= THRESHOLD_KM)
dir_u = np.zeros((N, N)); dir_v = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            de = (lons[j]-lons[i]) * np.cos(np.radians(lats[i]))
            dn = lats[j]-lats[i]; nr = np.hypot(de, dn)
            if nr > 0:
                dir_u[i, j], dir_v[i, j] = de/nr, dn/nr
print(f'N={N}  T={T}  dyn_adj={dyn_adj.shape}')

N=37  T=19921  dyn_adj=(19921, 37, 37)


In [19]:
# identical feature recipe to 09_model_v2 cell v2feat
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at','u_wind','v_wind']
g = df.groupby('Station', sort=False)
df['wspd'] = np.hypot(df['u_wind'], df['v_wind'])
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

built = (BASE_FEATS +
    ['wspd','pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
assert built == FEATURE_COLS, 'feature order does not match v2_config.json'
TIME_IDX = [FEATURE_COLS.index(c) for c in TIME_COLS]

X_full = np.zeros((T, N, F), dtype=np.float32)
for fi, col in enumerate(FEATURE_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    X_full[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))
Y_raw = X_full[:, :, 0].copy()                                   # raw PM2.5 ug/m3
print(f'X_full {X_full.shape}  NaNs {int(np.isnan(X_full).sum())}  (assert passed: feature order matches v2)')

X_full (19921, 37, 26)  NaNs 0  (assert passed: feature order matches v2)


In [20]:
feat_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v2_feature_scaler.pkl'), 'rb'))
log_scaler  = pickle.load(open(os.path.join(SCALER_DIR, 'v2_log_scaler.pkl'), 'rb'))
assert feat_scaler.mean_.shape[0] == F

Xs_full = feat_scaler.transform(X_full.reshape(-1, F)).reshape(T, N, F).astype(np.float32)
def from_log(a):
    return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):
    return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Yl_full = to_log(Y_raw)
del X_full

clim_grid = np.load(CLIM_GRID_PATH)                              # (12, 24, N, 2), train-only wind clim
def compute_At_vec(u, v, eps=1e-6):
    al = u[:, None]*dir_u + v[:, None]*dir_v
    A = np.where(within_threshold & (al > 0.0), al/(dist_matrix + eps), 0.0).astype(np.float32)
    rs = A.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return A / rs
mo_arr = pd.DatetimeIndex(timestamps).month.values - 1
ho_arr = pd.DatetimeIndex(timestamps).hour.values
dyn_clim = np.zeros((T, N, N), dtype=np.float32)
for t in range(T):
    uv = clim_grid[mo_arr[t], ho_arr[t]]
    dyn_clim[t] = compute_At_vec(uv[:, 0], uv[:, 1])
print(f'scaled with saved v2 scalers | dyn_clim {dyn_clim.shape}')

scaled with saved v2 scalers | dyn_clim (19921, 37, 37)


In [21]:
# WATGCNv2 - identical to 09_model_v2 cell v2model
class WATGCNv2(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads,
                 forecast_horizon, n_stations, dropout=0.2, n_time=7, gcn_hops=2):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.time_enc = nn.Sequential(nn.Linear(n_time, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH = H, forecast_horizon

    @staticmethod
    def _norm_adj(A, n, device):
        I = torch.eye(n, device=device).unsqueeze(0).unsqueeze(0)
        A = A + I
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, TFut, Anchor):
        B, Tin, n, _ = X.shape
        h = self.act(self.input_proj(X))
        A = self._norm_adj(At, n, X.device)
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf)
        gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        Afc_n = self._norm_adj(Afc, n, X.device)
        ctx_fc = torch.matmul(Afc_n, ctx.unsqueeze(1).expand(B, self.FH, n, self.H))
        t_emb = self.time_enc(TFut).unsqueeze(2).expand(B, self.FH, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, t_emb], dim=-1)).squeeze(-1)
        anchor = Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)
        return resid + anchor

models = []
for sd in SEEDS:
    m = WATGCNv2(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(TIME_IDX), GCN_HOPS).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v2_seed{sd}.pt'), map_location=DEVICE))
    m.eval(); models.append(m)
print(f'loaded {len(models)} v2 checkpoints')

loaded 7 v2 checkpoints


In [22]:
train_end_idx = int(T * 0.70); val_end_idx = int(T * 0.85)
n_test = (T - val_end_idx) - (INPUT_WINDOW + HORIZON) + 1
test_g = np.arange(val_end_idx, val_end_idx + n_test)            # global window-start indices
print(f'test windows: {n_test}  (07 used 2918)')

BATCH = 32
preds_log = np.zeros((n_test, HORIZON, N), dtype=np.float32)
t0 = time.time()
with torch.no_grad():
    for b0 in range(0, n_test, BATCH):
        bi = test_g[b0:b0+BATCH]
        X   = np.stack([Xs_full[gg:gg+INPUT_WINDOW]                       for gg in bi])
        At  = np.stack([dyn_adj[gg:gg+INPUT_WINDOW]                       for gg in bi])
        Afc = np.stack([dyn_clim[gg+INPUT_WINDOW:gg+INPUT_WINDOW+HORIZON] for gg in bi])
        tf  = np.stack([Xs_full[gg+INPUT_WINDOW:gg+INPUT_WINDOW+HORIZON, 0, :][:, TIME_IDX] for gg in bi])
        anc = np.stack([Yl_full[gg+INPUT_WINDOW-1]                        for gg in bi])
        X, At, Afc, tf, anc = [torch.from_numpy(z.astype(np.float32)).to(DEVICE) for z in (X, At, Afc, tf, anc)]
        ens = torch.stack([mm(X, At, Afc, tf, anc) for mm in models], 0).mean(0)
        preds_log[b0:b0+len(bi)] = ens.cpu().numpy()
preds_real = from_log(preds_log)
print(f'inference {time.time()-t0:.0f}s | preds_real {preds_real.shape} | '
      f'range [{preds_real.min():.1f}, {preds_real.max():.1f}] ug/m3')

# HOW TO CHECK: n_test == 2918. predicted range roughly 5-600 ug/m3, no NaN.

test windows: 2918  (07 used 2918)
inference 8s | preds_real (2918, 24, 37) | range [3.2, 687.4] ug/m3


In [23]:
os.makedirs(DASH, exist_ok=True)

# --- stations.json ---
stations_export = [{'id': i, 'name': n,
                    'lat': float(df_st.iloc[i]['latitude']),
                    'lon': float(df_st.iloc[i]['longitude'])}
                   for i, n in enumerate(station_names)]
json.dump(stations_export, open(os.path.join(DASH, 'stations.json'), 'w'), indent=2)

# --- playback.json  (same schema / 'now' convention as notebook 08) ---
now_ts   = timestamps[val_end_idx + INPUT_WINDOW : val_end_idx + INPUT_WINDOW + n_test]
raw_last = np.stack([Y_raw[gg+INPUT_WINDOW-1]                       for gg in test_g])   # (n_test, N)
actual   = np.stack([Y_raw[gg+INPUT_WINDOW:gg+INPUT_WINDOW+HORIZON] for gg in test_g])   # (n_test, 24, N)

def aqi_category(v):
    if v <= 30:  return 'Good'
    if v <= 60:  return 'Satisfactory'
    if v <= 90:  return 'Moderate'
    if v <= 120: return 'Poor'
    if v <= 250: return 'Very Poor'
    return 'Severe'

sample_idx = np.linspace(0, n_test - 1, N_PLAYBACK).astype(int)
playback = []
for idx in sample_idx:
    e = {'window_idx': int(idx),
         'timestamp': pd.Timestamp(now_ts[idx]).strftime('%Y-%m-%d %H:%M'),
         'stations': []}
    for s in range(N):
        a24 = [round(float(x), 1) for x in actual[idx, :, s]]
        p24 = [round(float(x), 1) for x in preds_real[idx, :, s]]
        cur = round(float(raw_last[idx, s]), 1)
        e['stations'].append({
            'station_id': s, 'current_pm25': cur, 'current_aqi': aqi_category(cur),
            'actual_24h': a24, 'predicted_24h': p24, 'peak_predicted': round(max(p24), 1),
            'hours_to_breach_cpcb': next((h+1 for h, v in enumerate(p24) if v > CPCB_THRESHOLD), None)})
    playback.append(e)
json.dump(playback, open(os.path.join(DASH, 'playback.json'), 'w'))

# --- summary.json  (same schema as notebook 08, numbers from v2_metrics.json) ---
vm = json.load(open(os.path.join(RESULTS_DIR, 'v2_metrics.json')))
summary = {
    'model_name': 'WA-TGCN v2 (7-seed ensemble)', 'n_stations': N,
    'test_windows_total': int(n_test), 'test_windows_exported': len(playback),
    'test_period': f"{pd.Timestamp(now_ts[0]).strftime('%Y-%m-%d')} to "
                   f"{pd.Timestamp(now_ts[-1]).strftime('%Y-%m-%d')}",
    'rmse': vm['rmse'], 'mae': vm['mae'], 'r2': vm['r2'],
    'rmse_by_horizon': vm['rmse_by_horizon'], 'rmse_by_regime': vm['rmse_by_regime'],
    'cpcb_threshold': CPCB_THRESHOLD, 'who_threshold': WHO_THRESHOLD}
json.dump(summary, open(os.path.join(DASH, 'summary.json'), 'w'), indent=2)

mb = os.path.getsize(os.path.join(DASH, 'playback.json')) / 1e6
print(f"stations.json  {len(stations_export)} stations")
print(f"playback.json  {len(playback)} windows  {mb:.1f} MB")
print(f"summary.json   RMSE {summary['rmse']} | R2 {summary['r2']} | {summary['test_period']}")
print('\ndashboard_export/ now serves WA-TGCN v2.  ->  streamlit run app.py')
print('v1 export preserved at dashboard_export_v1/')

stations.json  37 stations
playback.json  970 windows  17.0 MB
summary.json   RMSE 60.92 | R2 0.6341 | 2025-12-08 to 2026-04-09

dashboard_export/ now serves WA-TGCN v2.  ->  streamlit run app.py
v1 export preserved at dashboard_export_v1/


In [24]:
# --- wind-coupling edges for the map: one A(t) slice per playback window ---
import json, os, numpy as np

val_end_idx = int(T * 0.85)
n_test = (T - val_end_idx) - (INPUT_WINDOW + HORIZON) + 1
sidx = np.linspace(0, n_test - 1, N_PLAYBACK).astype(int)
EDGE_MIN, EDGE_CAP = 0.03, 120

we = {}
for gi in sidx:
    g = val_end_idx + int(gi)
    A = dyn_adj[g + INPUT_WINDOW - 1]                 # graph at "now" (last input hour)
    ii, jj = np.where(A > EDGE_MIN)                   # edge i -> j : air moves from i to j
    ww = A[ii, jj]
    keep = np.argsort(ww)[::-1][:EDGE_CAP]
    we[str(int(gi))] = [[int(ii[k]), int(jj[k]), round(float(ww[k]), 3)] for k in keep]

path = os.path.join(DASH, "wind_edges.json")
json.dump(we, open(path, "w"), separators=(",", ":"))
print(f"saved {path}  {os.path.getsize(path)/1e6:.2f} MB  {len(we)} windows  "
      f"avg {np.mean([len(v) for v in we.values()]):.0f} edges/window")

saved D:\FYP\Eco-Twin\v2_clean\dashboard_export\wind_edges.json  1.56 MB  970 windows  avg 120 edges/window
